# 국민은행_토스앱 리뷰 1년치 수집_저장_텍스트분석_mini_project
* 1. google play에서 국민은행 스타뱅킹앱, 토스앱의 사용자 리뷰를 1년치 수집
* 2. 수집한 리뷰를 mysql에 저장
* 3. 수집된 데이터로 긍정/부정 리뷰 비율 분석(막대 그래프 비교)
* 4. 워드클라우드 분석
* 5. LDA 토픽 모델링 분석(최적 k 탐색 포함)
* 6. word2vec 생성 후 T-sne 로 시각화
* 7. 분석 결과를 통해 얻을 수 있는 인사이트(문제점, 개선방안) 보고서 작성

1. google play에서 국민은행 스타뱅킹앱, 토스앱의 사용자 리뷰를 1년치 수집

In [1]:
#!pip install selenium webdriver-manager

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import pandas as pd
import time
from datetime import datetime, timedelta

In [3]:
# 앱 페이지 열고 리뷰 전체 보기 클릭

def create_driver(app_id):

    url = f"https://play.google.com/store/apps/details?id={app_id}&hl=ko&gl=KR"

    options = Options()
    options.add_argument("--start-maximized")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

    driver.get(url)

    wait = WebDriverWait(driver, 10)

    review_btn = wait.until(
        EC.element_to_be_clickable((By.XPATH, "//span[text()='리뷰 모두 보기']"))
    )

    review_btn.click()

    return driver

In [16]:
def get_reviews(driver, days):
    review_list = []
    limit = datetime.today() - timedelta(days=days)
    
    # 리뷰들이 로딩될 때까지 기다림
    wait = WebDriverWait(driver, 10)
    
    while True:
        # 리뷰 상자들 찾기
        reviews = driver.find_elements(By.CSS_SELECTOR, ".RHo1pe")
        
        for review in reviews:
            try:
                # 1. 날짜 확인
                date_str = review.find_element(By.CSS_SELECTOR, ".bp9Aid").text
                clean_date = date_str.replace(" ","").replace("년",".").replace("월",".").replace("일","")
                review_date = datetime.strptime(clean_date, "%Y.%m.%d")

                if review_date < limit:
                    return review_list

                # 2. 별점 확인 (이 부분이 에러가 많이 남)
                star_label = review.find_element(By.CSS_SELECTOR, ".iP21be").get_attribute("aria-label")
                # "별표 5개 만점에 5개를 받았습니다" -> 여기서 숫자만 추출
                rating = int(''.join(filter(str.isdigit, star_label.split(' ')[-1])))

                # 3. 리뷰 텍스트
                text = review.find_element(By.CSS_SELECTOR, ".h3YV2d").text

                review_list.append((text, date_str, rating))
            except:
                continue

        # 스크롤 내리기
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        print(f"\r현재 {len(review_list)}개 수집 중...", end="")
        
        # 더이상 내릴 곳이 없으면 중단
        # (중략 - 기존 코드와 동일)

In [17]:
# 리뷰 저장 함수
# 별점(rating)이 추가되었으므로 저장 함수도 컬럼명을 맞춰줘야 합니다.
# 화면 출력을 위해 return df 를 추가

def review_extractnsave(items, app):
    # 과제 3번을 위해 "rating" 컬럼도 수집 리스트에 포함되어 있어야 합니다.
    # 만약 리스트가 (text, date) 2개라면 columns=["review", "date"]로 유지하세요.
    df = pd.DataFrame(items, columns=["review", "date", "rating"]) 

    df["app"] = app

    # 파일 저장
    df.to_csv(f"{app}_reviews.csv", index=False, encoding="utf-8-sig")
    
    print(f"--- {app} 수집 및 저장 완료 ---")
    return df # 수집된 데이터를 반환해서 화면에 띄울 수 있게 함

In [18]:
# 실행 코드 - 금융앱 2개만 필요
# 결과 확인용 리스트 생성

urls = ["viva.republica.toss", "com.kbstar.kbbank"]
collected_dfs = [] # 수집된 데이터프레임들을 담을 바구니

for url in urls:
    driver = create_driver(url)
    
    # 수집 시작 (1년치)
    items = get_reviews(driver, 365)
    
    # 저장함수에서 반환된 df를 변수에 담기
    result_df = review_extractnsave(items, url)
    collected_dfs.append(result_df)
    
    driver.quit()

# 수집된 모든 데이터를 합쳐서 화면에 출력
final_df = pd.concat(collected_dfs, ignore_index=True)
display(final_df) # 주피터 노트북에서 표 형태로 출력됨

현재 0개 수집 중...

InvalidSessionIdException: Message: invalid session id: session deleted as the browser has closed the connection
from disconnected: not connected to DevTools
  (Session info: chrome=145.0.7632.117); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0xdc7dd3
	0xdc7e14
	0xbd1db0
	0xbc0d4e
	0xbdf895
	0xc452ec
	0xc5b0d9
	0xc3e7d6
	0xc10049
	0xc10e04
	0x1026924
	0x1021bf7
	0x103f5a0
	0xde0f58
	0xde891d
	0xdd0648
	0xdd0812
	0xdba21a
	0x74b45d49
	0x770ad83b
	0x770ad7c1


2. 수집한 리뷰를 mysql에 저장

In [10]:
import pymysql

def to_mysql(df, app_name):
    # 1. DB 접속 (비번 1234 맞는지 확인!)
    conn = pymysql.connect(
        host='localhost', user='root', password='1234', charset='utf8mb4'
    )
    cursor = conn.cursor()

    # 2. 데이터베이스(박스) 만들기
    cursor.execute("CREATE DATABASE IF NOT EXISTS bank_db")
    cursor.execute("USE bank_db")

    # 3. 테이블(표) 만들기 (review, date, rating, app 컬럼 생성)
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS reviews (
            review TEXT, date VARCHAR(50), rating INT, app VARCHAR(50)
        )
    """)

    # 4. 데이터프레임(df) 내용을 한 줄씩 DB에 넣기
    sql = "INSERT INTO reviews (review, date, rating, app) VALUES (%s, %s, %s, %s)"
    for i, row in df.iterrows():
        cursor.execute(sql, (row['review'], row['date'], row['rating'], app_name))

    conn.commit()
    conn.close()
    print(f">> {app_name} MySQL 저장 완료!")

In [9]:
import pandas as pd

# 저장된 파일 읽어오기
toss_df = pd.read_csv("viva.republica.toss_reviews.csv")
kb_df = pd.read_csv("com.kbstar.kbbank_reviews.csv")

# 두 데이터를 하나로 합치기
final_df = pd.concat([toss_df, kb_df], ignore_index=True)

# 결과 출력 (상위 10개만 확인)
print(f"전체 수집된 리뷰 개수: {len(final_df)}개")
display(final_df.head(10))

전체 수집된 리뷰 개수: 0개


,review,date,rating,app


3. 수집된 데이터로 긍정/부정 리뷰 비율 분석(막대 그래프 비교)

In [14]:
# 1. 라이브러리는 보내주신 파일 그대로 사용
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import koreanize_matplotlib 

# 2. 데이터가 비어있는지 먼저 확인 (매우 중요!)
if final_df.empty:
    print("데이터프레임(final_df)이 비어있습니다. 수집 코드를 먼저 확인하세요.")
else:
    # 별점이 문자열일 경우를 대비해 숫자로 변환
    final_df['rating'] = pd.to_numeric(final_df['rating'])

    # 3. 긍정/부정 라벨링 (보내주신 파일 스타일 응용)
    def labeling(rating):
        if rating >= 4: return '긍정'
        elif rating <= 2: return '부정'
        else: return '중립'

    final_df['label'] = final_df['rating'].apply(labeling)

    # 중립 제외하고 데이터가 있는지 확인
    analysis_df = final_df[final_df['label'] != '중립']
    
    if analysis_df.empty:
        print("긍정(4~5점)이나 부정(1~2점) 데이터가 하나도 없습니다. 현재 별점을 확인해 보세요.")
        print(final_df['rating'].value_counts()) # 현재 별점 분포 출력
    else:
        # 4. 앱별/라벨별 개수 계산
        result = analysis_df.groupby(['app', 'label']).size().reset_index(name='count')

        # 5. 그래프 그리기
        plt.figure(figsize=(10, 6))
        
        # hue(범례)가 명확히 나오도록 지정
        ax = sns.barplot(data=result, x='app', y='count', hue='label', 
                         palette={'긍정': '#3498db', '부정': '#e74c3c'})

        plt.title('금융 앱별 긍정/부정 리뷰 수 비교', fontsize=15)
        plt.xlabel('앱 이름', fontsize=12)
        plt.ylabel('리뷰 개수', fontsize=12)
        
        # 데이터가 있을 때만 범례 표시
        plt.legend(title='리뷰 성향')
        plt.grid(axis='y', linestyle='--', alpha=0.7)

        plt.show()

데이터프레임(final_df)이 비어있습니다. 수집 코드를 먼저 확인하세요.
